<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Run6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 2 CPECOG2: Coral bleaching, Run 6 (stronger backbone, larger input, cross-validated selection)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (about 50 to 60 min).

Why Run 6:

- **Ranking, not the cutoff, limits the score.** Run 5's test ROC AUC was about 0.905. At that level, macro F1 and bleached recall cannot both reach 0.90: a lower cutoff only trades one for the other. Run 6 tests changes that can raise ROC AUC: a stronger backbone and a larger input.
- **The split is the cleaned one again (629/132/131).** Run 5 merged on the original `splits.csv`, which put back the 9 photos filed under both labels and the test image that nearly duplicates a training image.
- **Selection uses 5-fold cross-validation** on the 761 training + validation images instead of one 132-image validation set, where one image moves macro F1 by about 0.75 points. Each candidate is scored on out-of-fold predictions for all 761 images, and the cutoff is chosen from them with the Run 3 rule (highest macro F1 with bleached recall of at least 0.90).
- **The test set is used once,** at the end, by the winner retrained on all 761 images with 3 seeds.

| Candidate | Backbone | Input | Notes |
| :-- | :-- | :-- | :-- |
| C1 | YOLOv8s-cls (ImageNet) | 224 | Run 5 R14 settings, the reference |
| C2 | YOLOv8s-cls (ImageNet) | 320 | most photos are 300 x 225 px, so 320 adds no detail; it gives the network a finer feature map |
| C3 | YOLOv8m-cls (ImageNet) | 224 | about 3 times the compute of YOLOv8s-cls, about 34 MB |
| C4 | DINOv2 ViT-S/14 | 224 | self-supervised on 142M images; often strong on small datasets; about 44 MB saved in half precision |

All YOLO candidates use the Run 5 R14 augmentation (dropout 0.3, RandAugment off, crop scale 0.25, flips, small color shifts). Run 5's `translate=0.1` is dropped: Ultralytics classification training never uses it, which is why R13 and R15 scored the same.

Outputs go to `results/run6/` and are zipped as `run6_results.zip`. The last cell prints a summary.

In [ ]:
!pip -q install ultralytics==8.4.211 imagehash kagglehub

In [ ]:
import os, glob, shutil, time, json, random, math, copy
import numpy as np, pandas as pd
from PIL import Image
import imagehash, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score, roc_curve, confusion_matrix

SEED = 42
FOLDS = 5
EPOCHS = 20                  # fixed epochs for every YOLO fit; no early stopping, the last epoch is used
DINO_EPOCHS = 15
BATCH = 32
FINAL_SEEDS = [42, 43, 44]
RECALL_MIN = 0.90            # Run 3 cutoff rule: highest macro F1 among cutoffs with bleached recall >= 0.90
MAX_MB, MAX_MS = 50, 200     # success criteria: model size and CPU latency per image
NEAR_DUP_T = 5
DATA_SRC = None              # set a folder path here to skip the Kaggle download
SPLITS_URL = "https://raw.githubusercontent.com/areslou/cpecog2-coral-bleaching/main/data/splits.csv"

# Run 5 R14 settings. translate is left out: Ultralytics classification augmentation does not use it.
YOLO_AUG = dict(dropout=0.3, auto_augment=None, erasing=0.0, scale=0.25, fliplr=0.5, flipud=0.5, hsv_h=0.0, hsv_s=0.2, hsv_v=0.1)
CANDIDATES = {
    "C1_yolov8s_224": dict(kind="yolo", weights="yolov8s-cls.pt", imgsz=224),
    "C2_yolov8s_320": dict(kind="yolo", weights="yolov8s-cls.pt", imgsz=320),
    "C3_yolov8m_224": dict(kind="yolo", weights="yolov8m-cls.pt", imgsz=224),
    "C4_dinov2_vits14_224": dict(kind="dino", imgsz=224),
}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU:", GPU_NAME if torch.cuda.is_available() else "none. Runtime > Change runtime type > T4 GPU, or this will take hours.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
OUT = os.path.join(BASE, "results", "run6"); os.makedirs(OUT, exist_ok=True)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
T_START = time.time()

## 1. Data: the cleaned split (same as Runs 2 and 3)

The Run 1 split from `data/splits.csv`, minus every copy of the 9 photos filed under both labels, minus the validation or test copy of near-duplicates that cross splits. Training and validation are then pooled (761 images) and divided into 5 stratified folds; the 131 test images stay aside.

In [ ]:
if DATA_SRC is None:
    import kagglehub
    DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent: return "bleached"
    if "health" in parent: return "healthy"
    return None

def phash_of(p):
    try:
        with Image.open(p) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def hamming(a, b):
    x = np.array([int(h, 16) for h in a], dtype=np.uint64)[:, None] ^ np.array([int(h, 16) for h in b], dtype=np.uint64)[None, :]
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POP[np.ascontiguousarray(x).view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1).astype(np.int16)

paths = sorted(p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True) if p.lower().endswith(exts))
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_of)
df["file"] = df["path"].map(os.path.basename)
df["phash"] = df["path"].map(phash_of)
df = df.dropna(subset=["label", "phash"]).reset_index(drop=True)

splits1 = pd.read_csv(SPLITS_URL).rename(columns={"path": "file"})
df = df.merge(splits1[["file", "label", "split"]], on=["file", "label"], how="left")
assert int(df["split"].notna().sum()) == len(splits1), "splits.csv did not match the download"

n_lab = df.groupby("phash")["label"].nunique()
conflict_hashes = set(n_lab[n_lab > 1].index)
clean = df[df["split"].notna() & ~df["phash"].isin(conflict_hashes)].reset_index(drop=True)
D = hamming(clean["phash"], clean["phash"])
sp, lb = clean["split"].to_numpy(), clean["label"].to_numpy()
ii, jj = np.where(np.triu(D <= NEAR_DUP_T, k=1))
rank = {"train": 0, "val": 1, "test": 2}
drop = {max((i, j), key=lambda k: rank[sp[k]]) for i, j in zip(ii, jj) if sp[i] != sp[j] and lb[i] == lb[j]}
drop |= {k for i, j in zip(ii, jj) if lb[i] != lb[j] for k in (i, j)}
clean = clean.drop(index=sorted(drop)).reset_index(drop=True)
print(f"Removed {len(conflict_hashes)} conflicting-label groups and {len(drop)} near-duplicates")
print(pd.crosstab(clean["split"], clean["label"], margins=True))

In [ ]:
def pad_square(p):
    im = Image.open(p).convert("RGB"); s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0)); sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    return sq

POOL = os.path.join(BASE, "coral_pool"); shutil.rmtree(POOL, ignore_errors=True)
imgs = []
for _, r in clean.iterrows():
    d = os.path.join(POOL, r["label"]); os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.splitext(r["file"])[0] + ".jpg")
    pad_square(r["path"]).save(dst, quality=95); imgs.append(dst)
clean["img"] = imgs
assert clean["img"].nunique() == len(clean), "file name clash in the image pool"

trval = clean[clean["split"] != "test"].reset_index(drop=True)
test = clean[clean["split"] == "test"].reset_index(drop=True)
y_trval = (trval["label"] == "bleached").astype(int).to_numpy()
y_test = (test["label"] == "bleached").astype(int).to_numpy()
test_files = test["img"].tolist()
FOLD_IDX = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(trval)), y_trval))
fold_of = np.zeros(len(trval), dtype=int)
for k, (_, va) in enumerate(FOLD_IDX): fold_of[va] = k
trval.assign(fold=fold_of)[["file", "label", "split", "fold"]].to_csv(os.path.join(OUT, "folds.csv"), index=False)
test[["file", "label"]].to_csv(os.path.join(OUT, "test_images.csv"), index=False)
print(f"Train + validation: {len(trval)} ({y_trval.sum()} bleached) in {FOLDS} folds | test: {len(test)} ({y_test.sum()} bleached)")

## 2. Helpers: metrics, the cutoff rule, and training for each model family

YOLO candidates train with Ultralytics on folder links for each fold, with a fixed number of epochs and no early stopping, so no held-out images steer training. DINOv2 ViT-S/14 is fine-tuned with a small PyTorch loop: a linear layer on the class token and the mean patch token, dropout 0.3, AdamW (backbone 1e-5, head 1e-3, weight decay 0.05), cosine schedule with a 1-epoch warm-up, mixed precision, random flips and crops of 87 to 100% of the side.

In [ ]:
def metrics_at(y, p, thr):
    yp = (np.asarray(p) >= thr).astype(int)
    return dict(macro_f1=f1_score(y, yp, average="macro"), bleached_recall=recall_score(y, yp, zero_division=0),
                bleached_precision=precision_score(y, yp, zero_division=0), healthy_recall=recall_score(y, yp, pos_label=0, zero_division=0),
                accuracy=accuracy_score(y, yp), roc_auc=roc_auc_score(y, p), cm=confusion_matrix(y, yp, labels=[0, 1]).tolist())

def pick_threshold(y, p):
    """Run 3 rule: the cutoff with the highest macro F1 among cutoffs that reach the recall floor."""
    rows = [(t, f1_score(y, p >= t, average="macro"), recall_score(y, p >= t, zero_division=0)) for t in np.round(np.linspace(0.01, 0.99, 981), 3)]
    ok = [r for r in rows if r[2] >= RECALL_MIN]
    return max(ok or rows, key=lambda r: (r[1], r[0]))[0], bool(ok)

def make_folder(name, train_idx, val_idx):
    """Ultralytics folder layout made of links to the image pool."""
    root = os.path.join(BASE, "cv", name); shutil.rmtree(root, ignore_errors=True)
    for part, idx in (("train", train_idx), ("val", val_idx)):
        for i in idx:
            r = trval.iloc[i]; d = os.path.join(root, part, r["label"]); os.makedirs(d, exist_ok=True)
            os.symlink(r["img"], os.path.join(d, os.path.basename(r["img"])))
    return root

from ultralytics import YOLO
def yolo_fit(cfg, data_root, name, seed):
    m = YOLO(cfg["weights"])
    m.train(data=data_root, epochs=EPOCHS, imgsz=cfg["imgsz"], batch=BATCH, seed=seed, deterministic=True, patience=EPOCHS + 1,
            val=False, plots=False, project=os.path.join(BASE, "runs6"), name=name, exist_ok=True, device=DEVICE, verbose=False, **YOLO_AUG)
    return str(m.trainer.last)

def yolo_predict(weights, files, imgsz, bs=64):
    m = YOLO(weights); b = next(k for k, n in m.names.items() if n == "bleached"); p = []
    for k in range(0, len(files), bs):
        p += [float(r.probs.data[b]) for r in m.predict(files[k:k + bs], imgsz=imgsz, device=DEVICE, verbose=False)]
    return np.array(p)

IMNET_MEAN, IMNET_STD = torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])

def load_dino_backbone():
    return torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", verbose=False)

class DinoClassifier(nn.Module):
    def __init__(self, backbone, n_classes=2, p=0.3):
        super().__init__()
        self.backbone, self.drop = backbone, nn.Dropout(p)
        self.head = nn.Linear(2 * backbone.embed_dim, n_classes)
    def forward(self, x):
        f = self.backbone.forward_features(x)
        return self.head(self.drop(torch.cat([f["x_norm_clstoken"], f["x_norm_patchtokens"].mean(1)], 1)))

_tensors = {}
def load_tensor(files, size):
    return torch.stack([torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))).permute(2, 0, 1)
                        for p in files])

def trval_tensor(size):
    if size not in _tensors: _tensors[size] = load_tensor(trval["img"].tolist(), size)
    return _tensors[size]

def norm(xb):
    return (xb.float() / 255 - IMNET_MEAN.to(xb.device)[:, None, None]) / IMNET_STD.to(xb.device)[:, None, None]

def augment(xb):
    if random.random() < 0.5: xb = xb.flip(3)
    if random.random() < 0.5: xb = xb.flip(2)
    n = xb.shape[-1]; c = int(n * random.uniform(0.87, 1.0)); i, j = random.randint(0, n - c), random.randint(0, n - c)
    return F.interpolate(xb[..., i:i + c, j:j + c], size=(n, n), mode="bilinear", align_corners=False)

def dino_fit(train_idx, seed, size):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    X, y = trval_tensor(size)[train_idx], torch.as_tensor(y_trval[train_idx])
    net = DinoClassifier(load_dino_backbone()).to(TDEV)
    opt = torch.optim.AdamW([{"params": net.backbone.parameters(), "lr": 1e-5}, {"params": net.head.parameters(), "lr": 1e-3}], weight_decay=0.05)
    steps_per = math.ceil(len(X) / BATCH); total = DINO_EPOCHS * steps_per
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / steps_per) * 0.5 * (1 + math.cos(math.pi * s / total)))
    use_amp = TDEV.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    for ep in range(DINO_EPOCHS):
        net.train(); perm = torch.randperm(len(X))
        for k in range(0, len(perm), BATCH):
            idx = perm[k:k + BATCH]
            if len(idx) < 2: continue
            xb = augment(norm(X[idx].to(TDEV)))
            with torch.autocast(device_type=TDEV.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(net(xb), y[idx].to(TDEV))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return net.eval()

def dino_predict(net, X, bs=64):
    out = []
    with torch.no_grad(), torch.autocast(device_type=TDEV.type, dtype=torch.float16, enabled=TDEV.type == "cuda"):
        for k in range(0, len(X), bs):
            out.append(net(norm(X[k:k + bs].to(TDEV))).float().softmax(1)[:, 1].cpu())
    return torch.cat(out).numpy()

## 3. Size and CPU-latency check before training

Each candidate's size (as saved, half precision) and CPU forward time (batch 1, after 5 warm-ups) are checked first, so no time is spent on a model that cannot meet the 50 MB and 200 ms limits. The final model's latency is measured again in section 7 with image loading included.

In [ ]:
def build_for_check(cfg):
    if cfg["kind"] == "yolo":
        return YOLO(cfg["weights"]).model.float().eval()
    return DinoClassifier(load_dino_backbone()).float().eval()

check, SKIP = [], []
for name, cfg in CANDIDATES.items():
    net = build_for_check(cfg).cpu()
    params = sum(p.numel() for p in net.parameters())
    x = torch.rand(1, 3, cfg["imgsz"], cfg["imgsz"])
    with torch.no_grad():
        for _ in range(5): net(x)
        t0 = time.perf_counter()
        for _ in range(20): net(x)
    ms = (time.perf_counter() - t0) / 20 * 1000
    mb = params * 2 / 1e6
    ok = mb < MAX_MB and ms < 0.9 * MAX_MS            # 10% margin for image loading
    check.append(dict(candidate=name, params_m=round(params / 1e6, 2), size_mb_fp16=round(mb, 1), cpu_forward_ms=round(ms), within_limits=ok))
    if not ok: SKIP.append(name)
    del net
check = pd.DataFrame(check); check.to_csv(os.path.join(OUT, "size_latency_check.csv"), index=False)
print(check.to_string(index=False))
print("Skipped (over a limit):", SKIP or "none")

## 4. 5-fold cross-validation

Every candidate is trained 5 times on the same folds; each image in the 761 gets a prediction from the model that did not train on it (out-of-fold). The folds are the same for every candidate, so the comparison is paired. Progress prints after each fold.

In [ ]:
FOLD_ROOTS = [make_folder(f"fold{k}", tr, va) for k, (tr, va) in enumerate(FOLD_IDX)]
oof, fold_auc, minutes = {}, {}, {}
for name, cfg in CANDIDATES.items():
    if name in SKIP: continue
    p, aucs, t0 = np.full(len(trval), np.nan), [], time.time()
    for k, (tr, va) in enumerate(FOLD_IDX):
        if cfg["kind"] == "yolo":
            w = yolo_fit(cfg, FOLD_ROOTS[k], f"{name}_f{k}", SEED)
            p[va] = yolo_predict(w, trval["img"].iloc[va].tolist(), cfg["imgsz"])
        else:
            net = dino_fit(tr, SEED, cfg["imgsz"])
            p[va] = dino_predict(net, trval_tensor(cfg["imgsz"])[va])
            del net; torch.cuda.empty_cache()
        aucs.append(roc_auc_score(y_trval[va], p[va]))
        print(f">>> {name} fold {k + 1}/{FOLDS}: ROC AUC {aucs[-1]:.3f} ({(time.time() - t0) / 60:.1f} min so far)")
    oof[name], fold_auc[name], minutes[name] = p, aucs, (time.time() - t0) / 60
    pd.DataFrame({"file": trval["file"], "true_label": y_trval, **{k: v for k, v in oof.items()}}).to_csv(os.path.join(OUT, "oof_predictions.csv"), index=False)

In [ ]:
rows = []
for name, p in oof.items():
    thr, reached = pick_threshold(y_trval, p)
    m5, mt = metrics_at(y_trval, p, 0.5), metrics_at(y_trval, p, thr)
    rows.append(dict(candidate=name, oof_roc_auc=m5["roc_auc"], fold_auc_mean=np.mean(fold_auc[name]), fold_auc_std=np.std(fold_auc[name], ddof=1),
                     macro_f1_at_0_5=m5["macro_f1"], recall_at_0_5=m5["bleached_recall"], cutoff=thr, recall_floor_reached=reached,
                     macro_f1_at_cutoff=mt["macro_f1"], recall_at_cutoff=mt["bleached_recall"], healthy_recall_at_cutoff=mt["healthy_recall"],
                     minutes=minutes[name]))
cv = pd.DataFrame(rows).sort_values(["recall_floor_reached", "macro_f1_at_cutoff", "oof_roc_auc"], ascending=False)
cv.to_csv(os.path.join(OUT, "cv_results.csv"), index=False)
print(cv.round(3).to_string(index=False))
WINNER, THR = cv.iloc[0]["candidate"], float(cv.iloc[0]["cutoff"])
print(f"\nWinner (out-of-fold macro F1 at the recall-floor cutoff): {WINNER}, cutoff {THR:.3f}")

fig, ax = plt.subplots(figsize=(5.2, 4.6))
markers = ["o", "s", "^", "D", "v", "P"]
for i, (name, p) in enumerate(oof.items()):
    fpr, tpr, _ = roc_curve(y_trval, p)
    ax.plot(fpr, tpr, color=COLORS[i % len(COLORS)], lw=2, marker=markers[i % len(markers)], markevery=0.1, ms=5,
            label=f"{name} (AUC {roc_auc_score(y_trval, p):.3f})")
ax.plot([0, 1], [0, 1], color="#888888", lw=1, ls="--")
ax.axhline(RECALL_MIN, color="#888888", lw=1, ls=":"); ax.text(0.55, RECALL_MIN - 0.05, "recall 0.90", fontsize=8, color="#555555")
ax.set_xlabel("False positive rate (healthy flagged as bleached)"); ax.set_ylabel("Bleached recall")
ax.set_title(f"Run 6 out-of-fold ROC, {FOLDS}-fold CV (n={len(trval)})", fontsize=10)
ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False); ax.legend(fontsize=7, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "cv_roc.png"), dpi=150); plt.show()

## 5. Final models: the winner retrained on all 761 images, 3 seeds

Same settings and number of epochs as in cross-validation. The cutoff is the one chosen from the out-of-fold predictions; the test set has not been used yet.

In [ ]:
cfg = CANDIDATES[WINNER]
FULL_ROOT = make_folder("full", np.arange(len(trval)), FOLD_IDX[0][1])   # the val folder only satisfies Ultralytics; it is never scored
final = []
for s in FINAL_SEEDS:
    t0 = time.time()
    if cfg["kind"] == "yolo":
        w = yolo_fit(cfg, FULL_ROOT, f"final_{WINNER}_s{s}", s)
        final.append(dict(seed=s, weights=w, predict=lambda files, w=w: yolo_predict(w, files, cfg["imgsz"])))
    else:
        net = dino_fit(np.arange(len(trval)), s, cfg["imgsz"])
        w = os.path.join(BASE, "runs6", f"final_{WINNER}_s{s}.pt"); os.makedirs(os.path.dirname(w), exist_ok=True)
        torch.save({k: v.half() for k, v in net.state_dict().items()}, w)
        final.append(dict(seed=s, weights=w, net=net, predict=lambda files, n=net: dino_predict(n, load_tensor(files, cfg["imgsz"]))))
    print(f">>> final seed {s}: {(time.time() - t0) / 60:.1f} min")

## 6. Held-out test set (evaluated once)

Each seed is scored at the cross-validated cutoff (the operating point) and at 0.5 (for comparison with earlier runs).

In [ ]:
rows, preds = [], {}
for f in final:
    p = f["predict"](test_files); preds[f["seed"]] = p
    pd.DataFrame(dict(file=test["file"], true_label=y_test, p_bleached=p)).to_csv(os.path.join(OUT, f"test_predictions_seed{f['seed']}.csv"), index=False)
    for name, t in (("cutoff", THR), ("0.5", 0.5)):
        rows.append(dict(seed=f["seed"], at=name, threshold=t, **metrics_at(y_test, p, t)))
test_res = pd.DataFrame(rows); test_res.to_csv(os.path.join(OUT, "test_results.csv"), index=False)
print(test_res.drop(columns=["cm"]).round(3).to_string(index=False))
TEST = {at: {c: (g[c].mean(), g[c].std(ddof=1) if len(g) > 1 else 0.0) for c in ["macro_f1", "bleached_recall", "healthy_recall", "accuracy", "roc_auc"]}
        for at, g in test_res.groupby("at")}
for at, m in TEST.items():
    print(f"At {at}:", {k: f"{v[0]:.3f} ± {v[1]:.3f}" for k, v in m.items()})

## 7. Size, CPU latency, and the model card

Latency includes image loading and preprocessing: mean over 100 test images, batch 1, after 10 warm-ups, on the CPU. Repeat it on a team laptop for the final report.

In [ ]:
f42 = final[0]
if cfg["kind"] == "yolo":
    FINAL_PATH = os.path.join(OUT, "final_model_s42.pt"); shutil.copy(f42["weights"], FINAL_PATH)
    mc = YOLO(FINAL_PATH)
    one = lambda p: mc.predict(p, imgsz=cfg["imgsz"], device="cpu", verbose=False)
else:
    FINAL_PATH = os.path.join(OUT, "final_model_s42_dinov2_vits14.pt"); shutil.copy(f42["weights"], FINAL_PATH)
    cpu_net = DinoClassifier(load_dino_backbone()).eval()
    cpu_net.load_state_dict({k: v.float() for k, v in torch.load(FINAL_PATH).items()})
    def one(p):
        x = torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((cfg["imgsz"], cfg["imgsz"]), Image.BILINEAR))).permute(2, 0, 1)[None]
        with torch.no_grad():
            return cpu_net(norm(x)).softmax(1)
for p in test_files[:10]: one(p)
t0 = time.perf_counter()
for p in test_files[:100]: one(p)
LAT_MS = (time.perf_counter() - t0) / len(test_files[:100]) * 1000
SIZE_MB = os.path.getsize(FINAL_PATH) / 1e6
card = dict(run=6, candidate=WINNER, architecture="DinoClassifier(dinov2_vits14) + linear on [CLS, mean patch]" if cfg["kind"] == "dino" else cfg["weights"],
            imgsz=cfg["imgsz"], classes={"0": "healthy", "1": "bleached"} if cfg["kind"] == "dino" else mc.names,
            preprocessing="pad to square with black, resize, RGB in [0,1]" + (", ImageNet mean/std" if cfg["kind"] == "dino" else ""),
            threshold=THR, threshold_source=f"{FOLDS}-fold out-of-fold predictions on train + val (recall >= {RECALL_MIN})",
            training=YOLO_AUG if cfg["kind"] == "yolo" else "AdamW backbone 1e-5 / head 1e-3, wd 0.05, cosine, flips + crops, dropout 0.3",
            epochs=EPOCHS if cfg["kind"] == "yolo" else DINO_EPOCHS, size_mb=round(SIZE_MB, 1), cpu_latency_ms=round(LAT_MS))
json.dump(card, open(os.path.splitext(FINAL_PATH)[0] + ".json", "w"), indent=2, default=str)
print(f"Final model: {SIZE_MB:.1f} MB, {LAT_MS:.0f} ms/image on CPU ({os.cpu_count()} cores)")

## 8. Summary for the progress log

In [ ]:
fmt = lambda ms: f"{ms[0]:.3f} ± {ms[1]:.3f}"
ok = lambda b: "met" if b else "NOT met"
lines = [f"RUN 6 SUMMARY ({time.strftime('%b. %d, %Y')})",
         f"Data: cleaned split; train + validation {len(trval)} ({int(y_trval.sum())} bleached) in {FOLDS} stratified folds; test {len(test)} ({int(y_test.sum())} bleached).",
         f"Skipped by the size/latency check: {SKIP or 'none'}.",
         f"Cross-validation (out-of-fold, n={len(trval)}):"]
for r in cv.itertuples():
    lines.append(f"  {r.candidate}: ROC AUC {r.oof_roc_auc:.3f} (folds {r.fold_auc_mean:.3f} ± {r.fold_auc_std:.3f}); macro F1 @0.5 {r.macro_f1_at_0_5:.3f}; "
                 f"cutoff {r.cutoff:.3f} -> macro F1 {r.macro_f1_at_cutoff:.3f}, bleached recall {r.recall_at_cutoff:.3f}, healthy recall {r.healthy_recall_at_cutoff:.3f}; {r.minutes:.1f} min")
lines += [f"Winner: {WINNER}; cutoff {THR:.3f} (from out-of-fold predictions).",
          f"Held-out test (n={len(test)}, evaluated once, seeds {FINAL_SEEDS}):",
          f"  At the cutoff: macro F1 {fmt(TEST['cutoff']['macro_f1'])}, bleached recall {fmt(TEST['cutoff']['bleached_recall'])}, "
          f"healthy recall {fmt(TEST['cutoff']['healthy_recall'])}, accuracy {fmt(TEST['cutoff']['accuracy'])}, ROC AUC {fmt(TEST['cutoff']['roc_auc'])}.",
          f"  At 0.5: macro F1 {fmt(TEST['0.5']['macro_f1'])}, bleached recall {fmt(TEST['0.5']['bleached_recall'])}.",
          "  Confusion matrices at the cutoff (rows true, cols predicted; healthy, bleached): "
          + "; ".join(f"seed {r.seed} {r.cm}" for r in test_res[test_res["at"] == "cutoff"].itertuples()),
          f"Size {SIZE_MB:.1f} MB; CPU latency {LAT_MS:.0f} ms/image (Colab CPU, batch 1, 100 test images after 10 warm-ups).",
          f"Targets on test (at the cutoff): macro F1 >= 0.90 {ok(TEST['cutoff']['macro_f1'][0] >= 0.90)}; "
          f"bleached recall >= 0.90 {ok(TEST['cutoff']['bleached_recall'][0] >= 0.90)}; size < 50 MB {ok(SIZE_MB < MAX_MB)}; latency < 200 ms {ok(LAT_MS < MAX_MS)}.",
          f"Environment: GPU {GPU_NAME}, ultralytics {__import__('ultralytics').__version__}, torch {torch.__version__}. Total time {(time.time() - T_START) / 60:.1f} min."]
summary = "\n".join(lines); print(summary)
open(os.path.join(OUT, "run6_summary.txt"), "w").write(summary)
ZIP = shutil.make_archive(os.path.join(BASE, "run6_results"), "zip", OUT)
print("\nSaved", ZIP)
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    pass